---
title: "Chapter 10: Content Standardization"
short_title: "10. Standardization"
---

Before the tree is built, the text of each version goes through the three standardization steps of Member 2. The
goal is that **the same content always yields the same string**, even when the authors format it differently across
versions; only then can the deduplication of [Chapter 12](./12_results.ipynb) match it exactly.

## 1. Removing formatting commands (`cleaner.py`)

| Group | Examples | Handling |
| :--- | :--- | :--- |
| Alignment, spacing, page breaks | `\centering`, `\vspace`, `\noindent`, `\newpage` | removed with their arguments |
| Table rules | `\hline`, `\toprule`, `\midrule`, `\cline` | removed |
| Font size, font style, color | `\small`, `\bfseries`, `\color` | removed |
| Wrapping commands | `\textbf{x}`, `\emph{x}`, `\underline{x}`, `{\bf x}` | `x` is kept |
| Float placement options | `\begin{figure}[htbp]` | becomes `\begin{figure}` |
| Layout environments | `center`, `minipage`, `multicols` | `\begin`/`\end` removed, content kept |

Commands that carry meaning are kept: sectioning, lists, `\cite`, `\ref`, `\label`, figure and table environments.

## 2. Math normalization (`normalizer.py`)

- **Inline math** `$...$`, `\(...\)`, `\begin{math}`, `\ensuremath{...}` all become `$...$`.
- **Display math** `$$...$$`, `\[...\]`, `equation*`, `displaymath` all become `\begin{equation}...\end{equation}`.
- Multi-line environments (`align`, `eqnarray`, `gather`, `multline`…) become an `equation` wrapping
  `aligned`/`gathered`, since an `equation` holds a single line; the formula itself, `\label` and `\tag` are kept.
- `\$`, `\verb`, `\url` and verbatim environments are not treated as math.

## 3. Sentence splitting and leaf extraction (`sentence_splitter.py`, `leaf_extractor.py`)

Each paragraph is split into leaf elements in reading order: **sentences**, **display formulas** and
**figures/tables**. A sentence ends at `.`, `!` or `?` followed by whitespace and the start of a new sentence. To
avoid wrong splits, the following are masked before splitting and restored afterwards: formulas, commands with
arguments (`\cite{a.b}`, `\ref{sec.1}`), URLs and DOIs, decimal numbers, and abbreviations (`e.g.`, `i.e.`,
`et al.`, `Fig.`, `Eq.`…).

## 4. Elements after standardization

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils_m2 import (COLORS, KIND_LABELS, PROBLEM_LABELS, group, human_bytes, human_seconds, load,
                             members, meta, metric, metric_table, num, ordered, pct, percent, ready, secs,
                             show, table)
READY = ready()

In [ ]:
if READY:
    kinds = [k for k in KIND_LABELS if k != "document"]
    rows = []
    for k in kinds:
        occ, uniq = metric(f"element_occurrences_kind_{k}"), metric(f"element_distinct_kind_{k}")
        rows.append({"kind": k, "Kind": KIND_LABELS[k], "Occurrences": occ, "Distinct elements": uniq})
    kt = pd.DataFrame(rows)
    kt = kt[kt["Occurrences"] > 0]
    long = kt.melt(id_vars=["Kind"], value_vars=["Occurrences", "Distinct elements"], var_name="Count", value_name="n")
    fig = px.bar(long, x="Kind", y="n", color="Count", barmode="group", log_y=True, color_discrete_sequence=COLORS,
                 title="Elements per kind: occurrences in every version and distinct elements",
                 labels={"n": "Count (log scale)"})
    show(fig)
    view = kt.copy()
    view["Reduction"] = (1 - view["Distinct elements"] / view["Occurrences"]).map(pct)
    for c in ("Occurrences", "Distinct elements"):
        view[c] = view[c].map(num)
    table(view.drop(columns="kind"), "Table 10.1. Elements per kind, whole group")

"Occurrences" counts every node of every version (one edge of the `hierarchy` table); "distinct elements" is the
number of rows of the `elements` table once identical elements across versions share one ID.

## 5. Leaf elements per version

In [ ]:
if READY:
    v = load("versions")
    tex = v[v["status"] == "ok"]
    long = tex.melt(id_vars=["member"], value_vars=["sentence", "formula", "figure"], var_name="kind", value_name="n")
    long["Kind"] = long["kind"].map(KIND_LABELS)
    long = long[long["n"] > 0]
    fig = px.box(long, x="Kind", y="n", color="Kind", log_y=True, color_discrete_sequence=COLORS,
                 title="Leaf elements per version with TeX (versions without that kind left out)",
                 labels={"n": "Elements (log scale)"})
    fig.update_layout(showlegend=False)
    show(fig)
    stats = []
    for k in ["heading", "sentence", "formula", "figure", "list", "nodes"]:
        s = tex[k]
        stats.append({"Kind": KIND_LABELS.get(k, "All nodes"), "Mean": num(s.mean(), 1), "Median": num(s.median(), 0),
                      "95th percentile": num(s.quantile(0.95), 0), "Maximum": num(s.max())})
    table(pd.DataFrame(stats), "Table 10.2. Nodes per version with TeX")

The book only shows counts; the sentences and formulas of the papers are not published before Milestone 3.